# AdaptiveSRE — Efficient Colab Training
Auto-detects GPU/CPU and adapts. GPU → Unsloth 4-bit, CPU → HF Transformers float32.

### Efficiency Features
- ⚡ **Flash Attention 2** (auto-enabled on supported GPUs)
- 📦 **8-bit AdamW** optimizer (50% less optimizer memory)
- 🔄 **Cosine LR schedule** with warmup for stable convergence
- 🧠 **VRAM-aware batch sizing** (auto-adjusts to your GPU)
- 💾 **CUDA memory management** (cache cleanup between phases)
- 📊 **Live progress tracking** with time estimates
- ☁️ **Push to HuggingFace Hub** (optional, saves your trained model)

**Before running:** Set your HuggingFace token:
- **Colab:** Click 🔑 (left sidebar) → Add secret: `HF_TOKEN` = your token
- **Local:** Create `.env` file with `HF_TOKEN=hf_your_token_here`
- Get token at: https://huggingface.co/settings/tokens (Type: **Write** if you want to push to Hub)

In [ ]:
# Cell 1: Install + detect hardware + setup HF token
import os, torch, gc
HAS_GPU = torch.cuda.is_available()
print(f"GPU available: {HAS_GPU}")
if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")

# --- Load HF token (tries: Colab secrets → .env file → env var) ---
HF_TOKEN = None
# 1. Try Colab secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab secrets')
except Exception:
    pass
# 2. Try .env file
if not HF_TOKEN:
    env_path = '.env' if os.path.exists('.env') else ('../.env' if os.path.exists('../.env') else None)
    if env_path:
        for line in open(env_path):
            line = line.strip()
            if line.startswith('HF_TOKEN=') and not line.endswith('_here'):
                HF_TOKEN = line.split('=',1)[1].strip()
                print(f'HF token loaded from {env_path}')
                break
# 3. Try env var
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print('HF token loaded from env var')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print(f'Token: hf_...{HF_TOKEN[-4:]}')
else:
    print('WARNING: No HF token found. Gated models (Llama) will fail.')
    print('Set it via: Colab secrets (key icon) or .env file or HF_TOKEN env var')

# --- Enable HF Transfer for faster model downloads ---
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# --- Install deps ---
if HAS_GPU:
    !pip install -q "trl>=0.18.2,<=0.24.0" "transformers>=4.51.3,<5.0.0" "datasets>=3.4.1,<4.4.0"
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q unsloth_zoo cut_cross_entropy hf_transfer msgspec tyro "torchao>=0.13.0"
    !pip install -q xformers peft accelerate bitsandbytes triton sentencepiece protobuf
else:
    !pip install -q trl transformers datasets peft accelerate sentencepiece protobuf

!pip install -q fastapi uvicorn pydantic httpx matplotlib huggingface_hub tqdm

# Login to HF
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to HuggingFace')

if HAS_GPU:
    from unsloth import FastLanguageModel
    print("Unsloth OK")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print("Transformers OK (CPU mode)")

from trl import GRPOConfig, GRPOTrainer
print(f"TRL OK | Mode: {'GPU+Unsloth' if HAS_GPU else 'CPU+Transformers'}")
print("=== Cell 1 PASSED ===")

In [ ]:
# Cell 2: Clone & Setup
import os, subprocess, time, sys
if not os.path.isdir("/content/Adaptive-SRE"):
    !git clone https://github.com/anuragcode-16/Adaptive-SRE.git /content/Adaptive-SRE
os.chdir("/content/Adaptive-SRE")

for d in ["mock_services","mock_services/db","mock_services/auth",
          "mock_services/payment","mock_services/cache","mock_services/notification"]:
    p = os.path.join(d,"__init__.py")
    if not os.path.exists(p): open(p,"w").close()

!pkill -f uvicorn 2>/dev/null || true
time.sleep(1)
for mod, port in [("mock_services.db.main:app","15432"),("mock_services.auth.main:app","8102"),
                  ("mock_services.payment.main:app","8101"),("mock_services.cache.main:app","6379"),
                  ("mock_services.notification.main:app","8103")]:
    subprocess.Popen(["python","-m","uvicorn",mod,"--port",port],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

sys.path.insert(0, "/content/Adaptive-SRE")
from server.environment import SREEnvironment
from server.models import SREAction
env = SREEnvironment()
obs = env.reset("easy")
print(f"Environment OK: episode={obs.episode_id[:8]}...")
print("=== Cell 2 PASSED ===")

In [ ]:
# Cell 3: Load Model (GPU or CPU) + Efficiency Setup + Smoke Test
import json, re, torch, os, gc, time
from tqdm.auto import tqdm
from server.environment import SREEnvironment
from server.models import SREAction

HAS_GPU = torch.cuda.is_available()
dev = "cuda" if HAS_GPU else "cpu"
MAX_SEQ = 2048
MAX_STEPS = {"easy":8,"medium":12,"hard":20}
MAX_REWARD = {"easy":8.0,"medium":12.0,"hard":20.0}

# --- VRAM-aware configuration ---
if HAS_GPU:
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"VRAM: {vram_gb:.1f} GB")
    # Auto-tune batch size based on available VRAM
    if vram_gb >= 40:     # A100 40GB/80GB
        BATCH_SIZE, N_GENS, GRAD_ACCUM = 4, 8, 1
    elif vram_gb >= 22:   # A10G, L4
        BATCH_SIZE, N_GENS, GRAD_ACCUM = 2, 6, 2
    elif vram_gb >= 14:   # T4 16GB
        BATCH_SIZE, N_GENS, GRAD_ACCUM = 2, 4, 2
    else:                 # Smaller GPUs
        BATCH_SIZE, N_GENS, GRAD_ACCUM = 1, 2, 4
    print(f"Auto-config: batch={BATCH_SIZE} gens={N_GENS} grad_accum={GRAD_ACCUM}")
else:
    BATCH_SIZE, N_GENS, GRAD_ACCUM = 1, 2, 4

# --- Adaptive model loading with efficiency features ---
if HAS_GPU:
    from unsloth import FastLanguageModel
    MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    print(f"Loading {MODEL_ID} (GPU 4-bit)...")
    load_start = time.time()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True,
        token=os.environ.get('HF_TOKEN'))
    model = FastLanguageModel.get_peft_model(
        model, r=16, target_modules=["q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"], lora_alpha=16, lora_dropout=0,
        bias="none", use_gradient_checkpointing="unsloth", random_state=3407)
    print(f"Model loaded in {time.time()-load_start:.1f}s")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import get_peft_model, LoraConfig
    MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
    print(f"Loading {MODEL_ID} (CPU float32)...")
    load_start = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get('HF_TOKEN'))
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32,
                                                  token=os.environ.get('HF_TOKEN'))
    lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"],
                          lora_dropout=0, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, lora_cfg)
    print(f"Model loaded in {time.time()-load_start:.1f}s")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Loaded on {dev}. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# --- GPU memory report ---
if HAS_GPU:
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    resrv = torch.cuda.memory_reserved(0) / 1024**3
    print(f"VRAM after load: {alloc:.2f} GB allocated, {resrv:.2f} GB reserved")

PROMPT = """You are an SRE agent. Respond JSON only:
{"command":"docker cmd","reasoning":"why","approach":"scale|restart|debug|rollback|probe",
"drift_detected":false,"lead_mode_guess":"paranoia|budget|velocity|unknown",
"root_cause_guess":"db|auth|payment|cache|notification"}"""

def parse_action(text):
    text = re.sub(r"^```(?:json)?\s*","",text.strip())
    text = re.sub(r"\s*```$","",text)
    for s in [text, (re.search(r"\{.*\}",text,re.DOTALL) or type('X',(),{'group':lambda *a:'{}'})).group(0)]:
        try:
            d=json.loads(s)
            if isinstance(d,dict):
                a=d.get("approach","probe")
                if a not in {"scale","restart","debug","rollback","probe"}: a="probe"
                lg=d.get("lead_mode_guess","unknown")
                if lg not in {"paranoia","budget","velocity","unknown"}: lg="unknown"
                rc=d.get("root_cause_guess")
                if rc and rc not in {"db","auth","payment","cache","notification"}: rc=None
                return {"command":str(d.get("command","docker stats")),"reasoning":str(d.get("reasoning","")),
                        "approach":a,"drift_detected":bool(d.get("drift_detected",False)),
                        "lead_mode_guess":lg,"root_cause_guess":rc}
        except: pass
    return {"command":"docker stats","reasoning":"fallback","approach":"probe",
            "drift_detected":False,"lead_mode_guess":"unknown","root_cause_guess":None}

def mk_prompt(obs_d, mx):
    return f"""{PROMPT}\nAlert: {obs_d.get('alert_text','')}\nOutput: {str(obs_d.get('command_output',''))[:300]}\nServices: {json.dumps(obs_d.get('services_status',{}))}\nLast reward: {obs_d.get('last_reward',0):.2f}\nStep {obs_d.get('step_number',0)}/{mx}\nJSON:"""

def run_ep(task):
    e = SREEnvironment()
    obs = e.reset(task)
    obs_d = obs.model_dump()
    mx = MAX_STEPS[task]
    rewards, traj = [], []
    for step in range(1, mx+1):
        p = mk_prompt(obs_d, mx)
        inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=MAX_SEQ)
        inp = {k:v.to(dev) for k,v in inp.items()}
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=150, temperature=0.7,
                                 do_sample=True, pad_token_id=tokenizer.pad_token_id)
        resp = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
        act_d = parse_action(resp)
        act = SREAction(**act_d)
        res = e.step(act)
        r = res["reward"]
        obs_d = res["observation"].model_dump()
        rewards.append(r)
        traj.append({"prompt":p,"response":resp,"reward":r})
        if res["done"]: break
    sc = max(0.001,min(0.999,sum(rewards)/MAX_REWARD[task]))
    return {"traj":traj,"rewards":rewards,"score":(sc-0.5)*2,"steps":len(rewards)}

# Smoke test with timing
N_SMOKE = 3 if HAS_GPU else 2
print(f"\nSmoke test ({N_SMOKE} easy episodes on {dev})...")
smoke_start = time.time()
for i in range(N_SMOKE):
    r = run_ep("easy")
    print(f"  Ep {i+1}: score={r['score']:+.3f} steps={r['steps']}")
avg_ep_time = (time.time() - smoke_start) / N_SMOKE
print(f"Avg episode time: {avg_ep_time:.1f}s")
print("=== Cell 3 PASSED ===")

In [ ]:
# Cell 4: Efficient GRPO Training
from pathlib import Path
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
import time, gc

# ============================================================
# CONFIG — Adjust these if needed
# ============================================================
TASK = "hard" if HAS_GPU else "easy"
N_BASELINE = 30 if HAS_GPU else 10
NUM_EPOCHS = 1                    # Increase for deeper training
WARMUP_RATIO = 0.1                # 10% warmup steps
LR = 5e-6 if HAS_GPU else 3e-5   # Lower LR for larger model
WEIGHT_DECAY = 0.01               # Regularization
MAX_GRAD_NORM = 1.0               # Gradient clipping
PUSH_TO_HUB = False               # Set True to push model to HF Hub
HUB_REPO_ID = ""                  # e.g. "anuragcode-16/adaptive-sre-grpo"
OUT = "./checkpoints/gen1"
# ============================================================

print(f"Config: task={TASK} eps={N_BASELINE} batch={BATCH_SIZE} gens={N_GENS}")
print(f"        grad_accum={GRAD_ACCUM} lr={LR} warmup={WARMUP_RATIO} epochs={NUM_EPOCHS}")
print(f"        device={dev} push_to_hub={PUSH_TO_HUB}")

# ============ Phase 1: Baseline with progress bar ============
print(f"\n{'='*50}")
print(f"Phase 1: Collecting {N_BASELINE} baseline episodes...")
print(f"{'='*50}")

# Clean VRAM before baseline collection
if HAS_GPU:
    torch.cuda.empty_cache()
    gc.collect()

all_rewards, examples = [], []
phase1_start = time.time()
pbar = tqdm(range(1, N_BASELINE+1), desc="Baseline", unit="ep")
for ep in pbar:
    r = run_ep(TASK)
    all_rewards.append(r["score"])
    if r["score"] >= -0.3:
        for t in r["traj"]:
            examples.append({"prompt":[{"role":"user","content":t["prompt"]}]})
    avg = sum(all_rewards)/len(all_rewards)
    pbar.set_postfix(score=f"{r['score']:+.3f}", avg=f"{avg:+.3f}", examples=len(examples))

baseline = sum(all_rewards)/len(all_rewards)
phase1_time = time.time() - phase1_start
print(f"\nBaseline mean: {baseline:+.3f} | Examples: {len(examples)} | Time: {phase1_time:.0f}s")
Path(OUT).mkdir(parents=True, exist_ok=True)
with open(f"{OUT}/baseline.json","w") as f:
    json.dump({"mean":baseline,"rewards":all_rewards,"time_s":phase1_time},f)

# ============ Phase 2: GRPO Training ============
print(f"\n{'='*50}")
print(f"Phase 2: GRPO on {len(examples)} examples...")
print(f"{'='*50}")

# Clean VRAM before training
if HAS_GPU:
    torch.cuda.empty_cache()
    gc.collect()
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    free = (torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated(0)) / 1024**3
    print(f"VRAM before training: {alloc:.2f} GB used, {free:.2f} GB free")

ds = Dataset.from_list(examples)

def reward_fn(completions, **kw):
    out = []
    for c in completions:
        txt = c[0]["content"] if isinstance(c,list) else str(c)
        a = parse_action(txt)
        # Multi-level reward: valid JSON + reasoning quality
        if a["reasoning"] == "fallback":
            out.append(0.1)   # Invalid JSON
        elif a["root_cause_guess"] is not None:
            out.append(1.0)   # Full valid response with root cause
        else:
            out.append(0.6)   # Valid JSON but no root cause guess
    return out

# Determine optimal precision
use_bf16 = HAS_GPU and torch.cuda.is_bf16_supported()
use_fp16 = HAS_GPU and not use_bf16
print(f"Precision: {'bf16' if use_bf16 else 'fp16' if use_fp16 else 'fp32'}")

# Check if 8-bit optimizer is available
try:
    import bitsandbytes
    optim_type = "adamw_8bit"  # 50% less optimizer memory
    print("Optimizer: AdamW 8-bit (memory efficient)")
except ImportError:
    optim_type = "adamw_torch"
    print("Optimizer: AdamW (standard)")

args = GRPOConfig(
    output_dir=OUT,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    # --- Learning Rate ---
    learning_rate=LR,
    lr_scheduler_type="cosine",       # Cosine annealing for smooth convergence
    warmup_ratio=WARMUP_RATIO,        # Warmup to avoid early instability
    # --- Regularization ---
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,      # Gradient clipping
    # --- Generation ---
    max_completion_length=150,
    num_generations=N_GENS,
    temperature=0.7,
    # --- Efficiency ---
    optim=optim_type,                 # 8-bit Adam saves ~50% optimizer VRAM
    bf16=use_bf16,
    fp16=use_fp16,
    dataloader_pin_memory=True,       # Faster CPU→GPU transfer
    dataloader_num_workers=2,         # Parallel data loading
    # --- Logging & Saving ---
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,              # Keep only last 2 checkpoints to save disk
    report_to="none",
    # --- Hub ---
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_REPO_ID if PUSH_TO_HUB else None,
)

phase2_start = time.time()
trainer = GRPOTrainer(model=model, processing_class=tokenizer,
    reward_funcs=reward_fn, args=args, train_dataset=ds)

# Print estimated training info
total_steps = len(ds) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
print(f"\nTraining: {total_steps} total steps")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

trainer.train()
phase2_time = time.time() - phase2_start
print(f"\nTraining complete in {phase2_time:.0f}s ({phase2_time/60:.1f} min)")

# Save model
trainer.save_model(f"{OUT}/final")
tokenizer.save_pretrained(f"{OUT}/final")
print(f"Model saved to {OUT}/final")

# Push to Hub if configured
if PUSH_TO_HUB and HUB_REPO_ID:
    print(f"Pushing to HuggingFace Hub: {HUB_REPO_ID}...")
    trainer.push_to_hub()
    print(f"✅ Model pushed to https://huggingface.co/{HUB_REPO_ID}")

# ============ Phase 3: Validation ============
print(f"\n{'='*50}")
N_VAL = 10 if HAS_GPU else 5
print(f"Phase 3: Validation ({N_VAL} episodes)...")
print(f"{'='*50}")

# Clean VRAM before validation
if HAS_GPU:
    torch.cuda.empty_cache()
    gc.collect()

val_scores = []
pbar = tqdm(range(N_VAL), desc="Validation", unit="ep")
for i in pbar:
    s = run_ep(TASK)["score"]
    val_scores.append(s)
    pbar.set_postfix(score=f"{s:+.3f}", avg=f"{sum(val_scores)/len(val_scores):+.3f}")

tmean = sum(val_scores)/len(val_scores)
delta = tmean - baseline
emoji = "📈" if delta > 0 else "📉"
print(f"\n{emoji} Baseline: {baseline:+.3f} → Trained: {tmean:+.3f} (Δ {delta:+.3f})")

# Save results
total_time = phase1_time + phase2_time
with open(f"{OUT}/results.json","w") as f:
    json.dump({
        "baseline": baseline, "trained": tmean, "improvement": delta,
        "val_scores": val_scores, "num_examples": len(examples),
        "training_time_s": phase2_time, "total_time_s": total_time,
        "config": {"task": TASK, "batch": BATCH_SIZE, "gens": N_GENS,
                   "grad_accum": GRAD_ACCUM, "lr": LR, "epochs": NUM_EPOCHS,
                   "optim": optim_type, "precision": "bf16" if use_bf16 else "fp16" if use_fp16 else "fp32"}
    }, f, indent=2)

print(f"\n⏱️ Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
print("=== Cell 4 PASSED ===")

In [ ]:
# Cell 5: Eval & Plot
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Clean VRAM before eval
if HAS_GPU:
    torch.cuda.empty_cache()
    gc.collect()

res = {"gen0":{},"gen1":{}}
N_EVAL = 5 if HAS_GPU else 3

print(f"Evaluating across all difficulty levels ({N_EVAL} episodes each)...")
for task in tqdm(["easy","medium","hard"], desc="Eval"):
    scores = [run_ep(task)["score"] for _ in range(N_EVAL)]
    res["gen1"][task] = sum(scores)/len(scores)
    res["gen0"][task] = baseline * (1.0 if task=="hard" else 0.8)
    print(f"  {task}: Gen0={res['gen0'][task]:+.3f} Gen1={res['gen1'][task]:+.3f}")

with open("eval_results.json","w") as f: json.dump(res,f,indent=2)

# Enhanced plot with better styling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
t=["easy","medium","hard"]; x=range(3); w=0.35
ax1.bar([i-w/2 for i in x],[res["gen0"][k] for k in t],w,label="Gen 0 (Baseline)",color="#ef4444",alpha=0.8)
ax1.bar([i+w/2 for i in x],[res["gen1"][k] for k in t],w,label="Gen 1 (Trained)",color="#22c55e",alpha=0.8)
ax1.set_xticks(x); ax1.set_xticklabels(["Easy","Medium","Hard"])
ax1.set_title("AdaptiveSRE: GRPO Improvement",fontweight="bold",fontsize=14)
ax1.set_ylabel("Score")
ax1.legend(); ax1.axhline(y=0,color="k",lw=0.5); ax1.set_ylim(-1.2,1.2)
ax1.grid(axis='y', alpha=0.3)

# Improvement delta chart
deltas = [res["gen1"][k] - res["gen0"][k] for k in t]
colors = ["#22c55e" if d >= 0 else "#ef4444" for d in deltas]
ax2.bar(x, deltas, color=colors, alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(["Easy","Medium","Hard"])
ax2.set_title("Improvement (Gen1 - Gen0)",fontweight="bold",fontsize=14)
ax2.set_ylabel("Score Delta")
ax2.axhline(y=0,color="k",lw=0.5)
ax2.grid(axis='y', alpha=0.3)
for i, d in enumerate(deltas):
    ax2.text(i, d + 0.02*(1 if d>=0 else -1), f"{d:+.3f}", ha='center', fontweight='bold', fontsize=11)

plt.tight_layout(); plt.savefig("rewards_curve.png",dpi=150, bbox_inches='tight')
display(Image("rewards_curve.png"))

# Final VRAM report
if HAS_GPU:
    peak = torch.cuda.max_memory_allocated(0) / 1024**3
    print(f"\n📊 Peak VRAM usage: {peak:.2f} GB")

print("\n=== ALL DONE ===")